# 23 - תחנות קריטיות בראי כל העדשות

כל מחברת בפרויקט הזה השיבה על השאלה *"אילו תחנות הן קריטיות?"* - וכל אחת מהן השיבה עליה אחרת. מחברת 03 קבעה שתחנה קריטית היא **cut vertex** (צומת חיתוך). מחברת 04 קבעה שמדובר בתחנה בעלת **betweenness** גבוה או **נפח שירות** גבוה. מחברת 05 קבעה שמדובר בתחנה שאין לה **חלופה בהליכה** בסביבתה. מחברות ההרחבה מוסיפות עוד: חשיבות **משוקללת ביקוש** (21), חשיבות **בשעת שיא** (20), **עלות זמן הנוסעים בעת סגירת התחנה** (22), ותפקיד **צומת מעבר רב-אופני** (17).

אלו אינן אותה שאלה, ואין שום הכרח שיתקבלו מהן אותן תשובות. תחנה כפרית יכולה להיות cut vertex ובה בעת לשרת שמונה נסיעות ביום; רציף במרכז העיר יכול לשרת אלפי נסיעות ביום ולהיות ניתן להחלפה בקלות מוחלטת מפני ששלושה רציפים זהים ניצבים ממול. המחברת הזו היא שלב הסינתזה: היא טוענת כל עדשה שזמינה בפועל, מדרגת את התחנות תחת כל אחת מהן, מודדת **עד כמה העדשות מסכימות ביניהן** (מתאם Spearman בין הדירוגים, מספר החפיפות ב-top-50, ומפות חום לשתי המדידות), ולאחר מכן מפרידה בין התחנות שהן קריטיות תחת **עדשות רבות** - הקריטיות באופן חסין - לבין תחנות שהן קריטיות תחת **עדשה אחת בלבד**, שהן על פי רוב תוצרי לוואי של הגדרת אותה עדשה ולא פגיעוּיות לאומיות אמיתיות.

התוצר הוא רשימה מקוצרת ומדורגת של תחנות קריטיות ברמה הלאומית, כאשר לכל אחת מהן מצורף נימוק מפורש לכשירותה. זהו הפריט שמתכנן תחבורה באמת יכול לפעול לפיו: לא עמודת מדד מרכזיות, אלא רשימה קצרה של שמות עם הצדקה צמודה לכל אחד מהם.

**שאלת המחקר הנדונה כאן:** האם ההגדרות השונות של "תחנה קריטית" שנעשה בהן שימוש לאורך הפרויקט מתכנסות לאותן תחנות, ואילו תחנות שורדות את כולן?

## קלט

המחברת היא **סובלנית לשחיקה**: רק הקלט הראשון הוא חובה, וכל עדשה אחרת נטענת אם המחברת שלה הורצה, ומדולגת עם הודעה מפורשת אם לא.

| עדשה | טבלת מקור | מחברת | נדרשת? |
|---|---|---|---|
| `betweenness` | `04*/tables/stop_metrics.csv` (`approx_betweenness`) | 04 | **כן** (גם מספקת את מרחב התחנות) |
| `service_volume` | `04*/tables/stop_metrics.csv` (`weighted_degree`) | 04 | **כן** |
| `articulation` | `03*/tables/articulation_points.csv` + `02*/tables/edges.csv` | 03, 02 | אופציונלית |
| `no_walk_alternative` | `05*/tables/critical_isolation.csv` (`nearest_alt_m`) | 05 | אופציונלית |
| `demand_weighted` | `21*/tables/demand_weighted_criticality.csv` | 21 | אופציונלית |
| `peak_hour` | `20*/tables/peak_vs_offpeak_centrality.csv` (+ `19*/tables/window_summary.csv`) | 20, 19 | אופציונלית |
| `closure_time_cost` | `22*/tables/rerouting_results.csv` | 22 | אופציונלית |
| `multimodal` | `17*/tables/transfer_hubs.csv` (`n_modes`) | 17 | אופציונלית |

**לא נקרא כאן שום GTFS גולמי.** הקובץ `stop_times.txt` אינו נוגע כלל, ולכן אין הורדה חיצונית ואין שלב streaming של 816 MB. הכול הוא join מעל טבלאות ששלבים קודמים כבר כתבו.

## פלט

הכול נכתב תחת `outputs/nb/23_critical_station_lenses/`:

* `tables/critical_station_lenses.csv` - פורמט ארוך, שורה אחת לכל (תחנה, עדשה): `stop_id, stop_name, lens, rank, score`.
* `tables/lens_agreement.csv` - שורה אחת לכל זוג עדשות: `lens_a, lens_b, spearman_rho, top50_overlap`.
* `tables/lens_agreement_detail.csv` - אותם זוגות בתוספת `n_common_stations`, כך שמתאם חלש שחושב על חפיפה דלילה לא ייחשב בטעות למתאם חזק.
* `tables/lens_coverage.csv` - מה כל עדשה מודדת, מהיכן הגיעה, וכמה מתוך 30k התחנות היא באמת מסוגלת לנקד.
* `tables/station_lens_profile.csv` - כל תחנה עם הדירוג והאחוזון שלה תחת כל עדשה.
* `tables/critical_station_shortlist.csv` - הרשימה המקוצרת הסופית והמדורגת המיועדת למתכנן, עם עמודת `reasons`.
* `tables/lens_specific_stations.csv` - תחנות שסומנו על ידי עדשה אחת בלבד (מועמדות לתוצרי לוואי).
* `lens_synthesis_summary.json` - מספרי הכותרת.
* `figures/lens_spearman_heatmap.png`, `lens_top_overlap_heatmap.png`, `stations_by_lens_count.png`, `shortlist_lens_profile.png`, `shortlist_map.png`, `lens_specific_counts.png`.

דבר מחוץ ל-`outputs/nb/23_critical_station_lenses/` אינו נכתב. התיקיות המצוטטות בדוח `outputs/tables`, `outputs/figures` ו-`outputs/rail` אינן נוגעות כלל.

## מחברות שחייבות לרוץ קודם

**חובה:** `04_centrality_analysis` (שדורשת בעצמה את `01` ואת `02`).
**מומלץ בחום:** `02_graph_construction` (עבור חומרת articulation מדויקת), `03_descriptive_analysis`, `05_critical_station_isolation`.
**אופציונלי, וזו הסיבה להריץ מחברת זו שוב בהמשך:** `17`, `19`, `20`, `21`, `22`. עם `02`-`05` בלבד המחברת עדיין רצה ומפיקה השוואה של ארבע עדשות; כל מחברת הרחבה שהורצה מוסיפה עמודה נוספת למטריצה.

## 1. אתחול סביבת העבודה

התא שלהלן מאפשר להריץ את המחברת הן על checkout מקומי והן על Google Colab. הוא מגדיר את `_ensure(...)`, שמתקין ב-pip רק את החבילות שחסרות בפועל (כך שהרצה חוזרת של המחברת זולה), ואת `find_repo_root()`, שמטפסת כלפי מעלה מהתיקייה הנוכחית בחיפוש אחר תיקיית ה-GTFS, ואם לא נמצאה - משכפלת את המאגר אל `/content`. לאחר מכן הוא מגדיר את `REPO`, `DATA` ו-`OUT` ויוצר את שורש הפלט של המחברת. כל תא מאוחר יותר מסתמך על שלושת הנתיבים האלה, ולכן תא זה חייב לרוץ ראשון.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. ספריות, תיקיות שלב וקבועים ניתנים לכוונון

אנו מייבאים את הסטאק המדעי ומקבעים את מבנה התיקיות: המחברת הזו הבעלים של `outputs/nb/23_critical_station_lenses/` על תת-התיקיות `tables/` ו-`figures/`, וקוראת כל שלב אחר לקריאה בלבד.

כל הכפתורים מרוכזים כאן כדי שבודק יוכל לשנות התנהגות במקום אחד:

* `TOP_K = 50` מגדיר מה פירוש "קריטית תחת עדשה" - הימצאות בתוך ה-top 50 של אותה עדשה. עמודת הפלט המחויבת על ידי חוזה הנתונים של הפרויקט נקראת מילולית `top50_overlap`, ולכן אם משנים את `TOP_K` שם העמודה חדל להתאים לתוכנה; המחברת מדפיסה אזהרה אם עושים זאת.
* `MIN_LENSES_FOR_SHORTLIST = 2` מונע מתחנה להגיע לרשימה המקוצרת מכוחה של עדשה יחידה שבמקרה היא היחידה המכסה אותה.
* `AP_SEVERITY_EXACT` הוא כפתור העלות היחיד הממשי. כאשר ערכו `True` המחברת מחשבת, עבור כל articulation point, כמה בדיוק תחנות נותרות מחוץ לרכיב השורד הגדול ביותר לאחר מחיקת אותה תחנה. זהו מעבר יחיד על הגרף לכל cut vertex - בערך 900 מעברים על גרף בן 30k צמתים / 52k קשתות, כלומר כ-**30-60 שניות** ב-Python טהור. קביעתו ל-`False` נסוגה לדירוג cut vertices לפי דרגה, שהוא מהיר אך פרוקסי גרוע יותר באופן מובהק, והמחברת אומרת זאת במפורש כשהיא עושה כן.
* `UNREACHABLE_PENALTY_SECONDS` רלוונטי רק אם מחברת 22 הורצה; ראו סעיף 11 לגבי מדוע הוא קיים ומדוע הוא הנחה ולא מדידה.

In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', font_scale=1.05)

STAGE = OUT / '23_critical_station_lenses'   # everything this notebook produces
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants ----------------------------------------------------
TOP_K = 50                          # "critical under a lens" = inside that lens's top K
MIN_LENSES_FOR_SHORTLIST = 2        # a station must be evaluated by >= this many lenses
SHORTLIST_SIZE = 40                 # rows in the final planner-facing shortlist
MIN_COMMON_FOR_RHO = 20             # skip Spearman when two lenses share fewer stations
AP_SEVERITY_EXACT = True            # exact cut-vertex severity (~30-60 s; see section 6)
UNREACHABLE_PENALTY_SECONDS = 3600  # cost charged per unit of unreachable demand (nb 22)
FIG_DPI = 150                       # figure resolution; drop to 90 for faster, smaller files
TOP_N_FIG = 25                      # rows in the profile heatmap and the bar figures

if TOP_K != 50:
    print(f'WARNING: TOP_K={TOP_K} but the contracted column name is "top50_overlap"; '
          'the column will hold top-' + str(TOP_K) + ' overlaps despite its name.')
print('this stage :', STAGE)

## 3. רינדור תוויות בעברית

שמות התחנות בקובץ ה-GTFS הישראלי הם בעברית, וכמה מהאיורים שלהלן מדפיסים אותם (במיוחד מפת החום של פרופיל הרשימה המקוצרת). Matplotlib אינה ממשת את אלגוריתם הדו-כיווניות (bidirectional) של Unicode, ולכן טקסט מימין לשמאל יוצא הפוך ובלתי קריא. התא שלהלן מבצע monkey-patch חד-פעמי ל-`matplotlib.text.Text.set_text` כך שכל מחרוזת המכילה תווים עבריים מומרת לסדר תצוגה באמצעות `python-bidi` לפני שהיא מצוירת, ובוחר גופן שיש בו למעשה גליפים עבריים (Arial ב-Windows, DejaVu Sans בכל מקום אחר). הוא אידמפוטנטי - הרצה חוזרת לא תערים patch על patch. כל שאר הטקסט במחברת הוא באנגלית, בהתאם לדרישת ההגשה.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. איתור שלבים קודמים

המחברת הזו קוראת עד שמונה שלבים אחרים, ומחציתם עשויים לא להתקיים עדיין. פונקציות העזר שלהלן מאתרות תיקיית שלב לפי **הקידומת הדו-ספרתית** שלה ולא לפי ה-slug המדויק (`OUT.glob('04*')`), כך שתיקיית שלב ששמה שונה מ-`04_centrality_analysis` ל-`04_centrality` עדיין מאותרת, והן מחפשות בתוכה רקורסיבית כך שאין חשיבות לשאלה אם השלב כתב את קבצי ה-CSV בשורש התיקייה או תחת `tables/`.

קיימות שתי פונקציות טעינה, וההבדל ביניהן מכוון:

* `require_table` זורקת `FileNotFoundError` הנוקבת בשם המחברת שיש להריץ תחילה. היא משמשת רק עבור מחברת 04, המספקת את מרחב התחנות שכל שאר העדשות מוטלות עליו.
* `optional_table` מדפיסה הודעת `[skipped]` בת שורה אחת הנוקבת בקובץ החסר ובמחברת שהייתה מייצרת אותו, רושמת את הפער ב-`MISSING_LENSES`, ומחזירה `None`. העדשה פשוט נעדרת מההשוואה - המחברת אינה ממציאה תחליף.

כל עמודת מזהה נאכפת ל-`str` בעת הקריאה. מזהי תחנות ב-GTFS הישראלי נראים מספריים, ולכן קריאת טבלה אחת ללא `dtype=str` הייתה יוצרת בשקט מפתחות שלמים וכל merge מולה היה חוזר ריק.

In [ ]:
# --- Stage resolution and loading helpers ---------------------------------
ID_COLUMNS = ['stop_id', 'from_stop', 'to_stop', 'removed_stop']
MISSING_LENSES = []      # (lens label, notebook that would provide it, what was missing)


def stage_dir(prefix):
    """Resolve a stage folder by two-digit prefix, e.g. '04' -> 04_centrality_analysis."""
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    return matches[0] if matches else None


def find_artifact(prefix, filename):
    """Full path of `filename` inside the stage folder with this prefix, or None."""
    folder = stage_dir(prefix)
    if folder is None:
        return None
    direct = folder / filename
    if direct.exists():
        return direct
    matches = sorted(folder.rglob(filename))
    return matches[0] if matches else None


def _read(path):
    """Read a stage CSV, forcing every known identifier column to string."""
    return pd.read_csv(path, dtype={c: str for c in ID_COLUMNS}, encoding='utf-8-sig')


def require_table(prefix, filename, notebook):
    """Load a mandatory upstream table, or fail with an actionable message."""
    path = find_artifact(prefix, filename)
    if path is None:
        raise FileNotFoundError(
            f'{filename} was not found under outputs/nb/{prefix}* - run notebook '
            f'{notebook} first; it is the stage that writes {filename}.')
    df = _read(path)
    print(f'[required] {filename:<38} {len(df):>7,} rows  <-  {path}')
    return df


def optional_table(prefix, filename, notebook, lens_label):
    """Load an optional upstream table; return None with a clear message if absent."""
    path = find_artifact(prefix, filename)
    if path is None:
        MISSING_LENSES.append((lens_label, notebook, filename))
        print(f'[skipped ] lens "{lens_label}" unavailable: {filename} not found under '
              f'outputs/nb/{prefix}* - run notebook {notebook} to enable it.')
        return None
    df = _read(path)
    print(f'[loaded  ] {filename:<38} {len(df):>7,} rows  <-  {path}')
    return df


print('stages currently present under', OUT)
for folder in sorted(p.name for p in OUT.iterdir() if p.is_dir()):
    print('   ', folder)

## 5. מרחב התחנות (עמוד השדרה)

כל עדשה חייבת להיות מוטלת על אותה קבוצת תחנות משותפת, אחרת "מקום 3 תחת עדשה A" ו"מקום 3 תחת עדשה B" אינם בני השוואה. אנו משתמשים ב-`stop_metrics.csv` של מחברת 04 כעמוד השדרה, משום שהוא מכסה בדיוק את התחנות הפעילות של גרף שכנוּת הנסיעות (כ-30,463 תחנות) ונושא את השם, הקואורדינטות והמחוז הדרושים לנו לאיורים ולרשימה המקוצרת הסופית.

עדשה עשויה לכסות פחות תחנות מעמוד השדרה - זה צפוי, וזו ההסתייגות החשובה ביותר במחברת. עדשת הבידוד ממחברת 05, למשל, העריכה רק תת-קבוצה מסוננת מראש של תחנות. כל תחנה שעדשה אינה מכסה נרשמת כ**לא הוערכה**, ולעולם לא כ"קיבלה ציון אפס": התייחסות לתחנה שלא הוערכה כאילו אינה חשובה הייתה ממירה בשקט פער כיסוי לממצא.

In [ ]:
# --- Station universe from notebook 04 ------------------------------------
metrics = require_table('04', 'stop_metrics.csv', '04_centrality_analysis')
metrics['stop_id'] = metrics['stop_id'].astype(str).str.strip()
metrics = metrics.drop_duplicates(subset='stop_id')

for needed in ('approx_betweenness', 'weighted_degree', 'stop_name', 'lat', 'lon'):
    if needed not in metrics.columns:
        raise KeyError(f'stop_metrics.csv has no "{needed}" column - re-run notebook 04; '
                       f'columns found: {list(metrics.columns)}')

keep = ['stop_id', 'stop_name', 'lat', 'lon', 'region']
keep = [c for c in keep if c in metrics.columns]
spine = metrics[keep].copy()
if 'region' not in spine.columns:
    spine['region'] = 'Unknown'

SPINE_IDS = set(spine['stop_id'])
NAME_BY_ID = dict(zip(spine['stop_id'], spine['stop_name']))
print(f'station universe: {len(spine):,} active stops')
spine.head()

## 6. מרשם העדשות

`register_lens` הוא המקום היחיד שבו עמודה גולמית משלב קודם הופכת לדירוג בר-השוואה. היא מקבלת טבלה, עמודת מזהה ועמודת ציון, ומבצעת חמישה דברים:

1. **מכוונת את הציון.** כל עדשה נשמרת כך ש*ציון גבוה יותר משמעו קריטי יותר*. עדשות שהמדד הטבעי שלהן מצביע בכיוון ההפוך (מספר דירוג, שבו 1 הוא הגרוע ביותר) מקבלות `higher_is_worse=False` והציון הנשמר עובר היפוך סימן. זו הסיבה שעמודת `score` בטבלה הארוכה המיוצאת היא ציון קריטיוּת ולא העתק מילולי של המדד המקורי - מוסכמת הסימן חייבת להיות אחידה כדי שלהשוואות שלהלן תהיה משמעות כלשהי.
2. **מסירה כפילויות** לכדי שורה אחת לתחנה (תוך לקיחת הערך הגרוע ביותר אם טבלה חוזרת על תחנה).
3. **מצמצמת לעמוד השדרה**, ומדווחת כמה שורות נאלצה להשמיט משום שהתחנה אינה בגרף שכנוּת הנסיעות.
4. **מדרגת** עם `method='min'`, כך שתחנות בתיקו חולקות דירוג - חשוב מכיוון שלכמה עדשות יש תיקואים ארוכים (ספירות אופנים, מעקפים בערכים שלמים קטנים).
5. **מחשבת אחוזון** של הציון בתוך קבוצת הכיסוי של העדשה עצמה, ומניבה ערך בתחום (0, 1] הבר-השוואה בין עדשות בעלות יחידות שונות לחלוטין (שניות, מטרים, נסיעות, betweenness חסר יחידות).

עדשה חסרה, ריקה או חסרת העמודה המבוקשת מדולגת עם הודעה ופשוט אינה מופיעה בהשוואה.

In [ ]:
# --- Lens registry --------------------------------------------------------
LENSES = {}      # lens name -> DataFrame[stop_id, score, rank, pct]
LENS_META = {}   # lens name -> provenance and coverage record


def register_lens(name, df, id_col, score_col, higher_is_worse=True,
                  question='', source=''):
    """Turn one upstream column into a ranked, percentiled lens over the spine."""
    if df is None:
        return None
    for col in (id_col, score_col):
        if col not in df.columns:
            print(f'[skipped ] lens "{name}": source table has no "{col}" column '
                  f'(found: {list(df.columns)[:8]} ...).')
            MISSING_LENSES.append((name, source, 'column ' + col))
            return None

    sub = df[[id_col, score_col]].copy()
    sub.columns = ['stop_id', 'score']
    sub['stop_id'] = sub['stop_id'].astype(str).str.strip()
    sub['score'] = pd.to_numeric(sub['score'], errors='coerce')
    if not higher_is_worse:
        sub['score'] = -sub['score']          # store so that higher always = more critical
    sub = sub.dropna(subset=['stop_id', 'score'])
    sub = sub.groupby('stop_id', as_index=False)['score'].max()

    before = len(sub)
    sub = sub[sub['stop_id'].isin(SPINE_IDS)]
    dropped = before - len(sub)
    if len(sub) < 2:
        print(f'[skipped ] lens "{name}": fewer than two usable stations after cleaning.')
        MISSING_LENSES.append((name, source, 'no usable rows'))
        return None

    sub['rank'] = sub['score'].rank(ascending=False, method='min').astype(int)
    sub['pct'] = sub['score'].rank(ascending=True, pct=True)
    sub = sub.sort_values('rank').reset_index(drop=True)

    LENSES[name] = sub
    LENS_META[name] = {
        'lens': name,
        'question': question,
        'source': source,
        'metric': score_col,
        'higher_metric_is_more_critical': bool(higher_is_worse),
        'stations_evaluated': int(len(sub)),
        'coverage_share_of_universe': round(len(sub) / len(SPINE_IDS), 4),
        'rows_dropped_outside_universe': int(dropped),
    }
    print(f'[lens ok ] {name:<20} {len(sub):>7,} stations '
          f'({len(sub) / len(SPINE_IDS):.1%} of the universe), metric = {score_col}')
    return sub

## 7. עדשה (א) betweenness טופולוגי ועדשה (ג) נפח שירות

שתיהן מגיעות ישירות ממחברת 04 ושתיהן מכסות את המרחב המלא, ולכן הן משמשות עוגן להשוואה.

* **`betweenness`** משתמשת ב-`approx_betweenness` - שיעור המסלולים הקצרים ביותר העוברים דרך התחנה. זהו המדד הקלאסי של "אם זו תיכשל, התנועה תיאלץ לעקוף אותה". שימו לב שהוא *מקורב*: מחברת 04 חישבה אותו בדגימת pivots, וטבלת היציבות שלה עצמה מראה שהדירוג יציב בראש אך רועש בזנב. יש להתייחס להפרשי דירוג של כמה מקומות כחסרי משמעות.
* **`service_volume`** משתמשת ב-`weighted_degree` - המספר הכולל של נסיעות מתוכננות הנוגעות בתחנה. זו הקריאה התפעולית ולא הטופולוגית של חשיבות, וזו העדשה היחידה כאן המשקפת כמה שירות באמת קיים ולא היכן התחנה יושבת בגרף.

השתיים מוחזקות בכוונה בנפרד ולא ממוזגות לעדשת "מרכזיות" אחת, משום שכל תכליתה של המחברת היא לבחון האם הן מסכימות.

In [ ]:
# --- Lens (a): topological betweenness ------------------------------------
register_lens(
    'betweenness', metrics, 'stop_id', 'approx_betweenness', higher_is_worse=True,
    question='Which stations carry the most shortest paths through the network?',
    source='04_centrality_analysis/tables/stop_metrics.csv')

# --- Lens (c): service volume ---------------------------------------------
register_lens(
    'service_volume', metrics, 'stop_id', 'weighted_degree', higher_is_worse=True,
    question='Which stations carry the largest scheduled trip volume?',
    source='04_centrality_analysis/tables/stop_metrics.csv')

## 8. עדשה (ב) articulation points, מדורגות לפי הנזק שהן באמת גורמות

מחברת 03 מונה את כ-900 ה-cut vertices, אך היות cut vertex היא תכונה **בינארית**, ותכונה בינארית אינה ניתנת לדירוג. רובם של אותם 900 הם צוואר של ענף מבוי סתום קצר: הסרתם מבודדת שלוש תחנות. קומץ מהם יושבים בין חלקים גדולים באמת של הרשת. השוואת דגל בינארי מול עדשה רציפה גם הייתה מרוקנת את מתאמי Spearman ממשמעות (משתנה אחד היה כולו תיקואים).

לכן אנו מחשבים חומרה לכל cut vertex: **כמה תחנות נותרות מחוץ לרכיב השורד הגדול ביותר כאשר אותה תחנה נמחקת**. הקוד בונה מילון שכנויות פשוט מתוך `02*/edges.csv` (הטלה לא מכוונת, לולאות עצמיות מוסרות), ועבור כל cut vertex מריץ מעבר depth-first איטרטיבי יחיד המתחיל מכל אחד משכניו כאשר הצומת עצמו מסומן כחסום. קבוצת הצמתים שבוקרו משותפת בין הענפים של צומת יחיד, כך שכל רכיב נסרק בדיוק פעם אחת והמעבר כולו עולה `O(V + E)` *עבור הרכיב של אותו צומת בלבד*.

**עלות:** בערך 900 מעברים על גרף בן 30k צמתים / 52k קשתות ב-Python טהור, כ-**30-60 שניות**. קביעת `AP_SEVERITY_EXACT = False` מדלגת על כך - המחברת אז מדרגת cut vertices לפי דרגה ומדפיסה אזהרה מפורשת, משום שדרגה היא תחליף גרוע לגודל הניתוק (תחנה בעלת דרגה 2 יכולה להיות החיבור היחיד לעיירה שלמה).

מרחב העדשה הזו הוא כ-900 ה-cut vertices, לא כל 30k התחנות. מתאמים המערבים אותה עונים אפוא על השאלה "בקרב cut vertices, האם גודל הניתוק עוקב אחרי שאר העדשות?" - שהיא השאלה הנכונה, אך צרה יותר מכפי שהיא נראית.

In [ ]:
# --- Lens (b): articulation points with an exact severance score ------------
ap_df = optional_table('03', 'articulation_points.csv', '03_descriptive_analysis',
                       'articulation')
edges_path = find_artifact('02', 'edges.csv')
ap_score_col = None

if ap_df is not None:
    ap_df = ap_df.copy()
    ap_df['stop_id'] = ap_df['stop_id'].astype(str).str.strip()

if ap_df is not None and edges_path is not None and AP_SEVERITY_EXACT:
    edges_df = _read(edges_path)
    adj = {}
    for u, v in zip(edges_df['from_stop'].astype(str), edges_df['to_stop'].astype(str)):
        if u == v:
            continue                     # self-loops carry no connectivity information
        adj.setdefault(u, set()).add(v)
        adj.setdefault(v, set()).add(u)
    print(f'undirected adjacency built: {len(adj):,} nodes')

    def _explore(start, visited):
        """Iterative DFS from `start`, skipping anything already in `visited`."""
        stack, members = [start], []
        visited.add(start)
        while stack:
            x = stack.pop()
            members.append(x)
            for y in adj[x]:
                if y not in visited:
                    visited.add(y)
                    stack.append(y)
        return members

    comp_size, _seen = {}, set()
    for node in adj:
        if node in _seen:
            continue
        members = _explore(node, _seen)
        for m in members:
            comp_size[m] = len(members)

    def stranded_nodes(v):
        """Stations left outside the largest surviving fragment when v is deleted."""
        blocked, sizes = {v}, []
        for start in adj[v]:
            if start in blocked:
                continue
            stack, count = [start], 0
            blocked.add(start)
            while stack:
                x = stack.pop()
                count += 1
                for y in adj[x]:
                    if y not in blocked:
                        blocked.add(y)
                        stack.append(y)
            sizes.append(count)
        return 0 if not sizes else sum(sizes) - max(sizes)

    ap_df = ap_df[ap_df['stop_id'].isin(adj)].copy()
    ap_df['stranded_nodes'] = [stranded_nodes(s) for s in ap_df['stop_id']]
    ap_df['component_size'] = [comp_size[s] for s in ap_df['stop_id']]
    ap_score_col = 'stranded_nodes'
    ap_df.sort_values('stranded_nodes', ascending=False).head(10).to_csv(
        TABLES / 'top_articulation_severity.csv', index=False, encoding='utf-8-sig')
    print(f'exact severance computed for {len(ap_df):,} cut vertices; '
          f'worst strands {int(ap_df["stranded_nodes"].max()):,} stations, '
          f'median strands {ap_df["stranded_nodes"].median():.0f}')

elif ap_df is not None:
    ap_score_col = 'degree' if 'degree' in ap_df.columns else None
    print('[fallback] ranking cut vertices by DEGREE, not by severance size, because '
          'edges.csv from notebook 02 is missing or AP_SEVERITY_EXACT is False. '
          'Degree is a weak proxy: a degree-2 stop can be the sole link to a whole town.')

if ap_df is not None and ap_score_col is not None:
    register_lens(
        'articulation', ap_df, 'stop_id', ap_score_col, higher_is_worse=True,
        question='Among cut vertices, which one strands the most stations when removed?',
        source='03_descriptive_analysis/tables/articulation_points.csv (+ 02 edges.csv)')

## 9. עדשה (ד) אין חלופה בהליכה

מחברת 05 מדדה, עבור כל תחנה שבחנה, את המרחק האווירי אל התחנה ה*אחרת* הקרובה ביותר שיכולה לקלוט את נוסעיה (`nearest_alt_m`). ערך גדול משמעו שלנוסע שנתקע שם אין לאן ללכת ברגל: התחנה אינה ניתנת להחלפה בשטח, ללא קשר למה שהגרף אומר.

**יש להיות כנים לגבי מה העדשה הזו מכסה.** מחברת 05 לא העריכה את כל כ-30k התחנות; היא העריכה קבוצה מסוננת מראש של תחנות שכבר סימנה כקריטיות (כ-3,000 שורות), ובתוכן, לרוב המכריע יש חלופה בטווח 300 מ'. לכן העדשה הזו היא *מותנית*: היא מדרגת חוסר יכולת להחלפה **בקרב תחנות שכבר נחשבו חשובות**, ואינה אומרת דבר כלל על כ-27k התחנות האחרות. משתי מסקנות נובעות מכך, ושתיהן מטופלות במפורש בהמשך: מתאמיה עם עדשות אחרות מחושבים על תת-הקבוצה הזו בלבד, ותחנה שנעדרת ממנה נרשמת כלא הוערכה ולא כ"יש לה חלופה".

In [ ]:
# --- Lens (d): distance to the nearest substitutable stop -------------------
iso = optional_table('05', 'critical_isolation.csv', '05_critical_station_isolation',
                     'no_walk_alternative')
if iso is not None:
    print(f'coverage note: notebook 05 evaluated {len(iso):,} pre-filtered stops, i.e. '
          f'{len(iso) / len(SPINE_IDS):.1%} of the universe - this lens is conditional '
          'on a station already having been flagged critical upstream.')
    if 'is_isolated' in iso.columns:
        flagged = int(pd.Series(iso['is_isolated']).astype(str).str.lower()
                      .isin(['true', '1', 'yes']).sum())
        print(f'   of those, {flagged:,} were flagged as genuinely isolated upstream.')

register_lens(
    'no_walk_alternative', iso, 'stop_id', 'nearest_alt_m', higher_is_worse=True,
    question='Which stations have no substitutable stop within walking distance?',
    source='05_critical_station_isolation/tables/critical_isolation.csv')

## 10. עדשה (ה) קריטיוּת משוקללת ביקוש

מחברת 21 מדרגת מחדש תחנות על ידי שילוב חשיבות טופולוגית עם האוכלוסייה שהתחנה משרתת, כך שתחנה מעניינת מבחינה מבנית באזור ריק יורדת, ותחנה שגרתית המשרתת 40,000 בני אדם עולה. אנו מעדיפים את עמודת `demand_weighted_rank` שלה (דירוג, שבו 1 הוא הקריטי ביותר - ומכאן `higher_is_worse=False`, ההופך את הסימן כך שהציון הנשמר שומר על המוסכמה הכלל-מחברתית). אם קיים רק הקובץ הביניים `demand_proxy.csv` אנו נסוגים לעמודת `demand_weight` שלו.

אם מחברת 21 לא הורצה, העדשה הזו מדולגת. אנו בכוונה **אין** מאלתרים תחליף מתוך הטבלה הסוציו-אקונומית של מחברת 08: אוכלוסייה המשויכת לתחנה באמצעות spatial join היא בדיוק הפרוקסי שמחברת 21 אחראית לבנות ולתקף, ובנייתו מחדש כאן תחת הנחות שונות הייתה מייצרת גרסה שנייה, בלתי עקבית בשקט, של אותו גודל עצמו.

In [ ]:
# --- Lens (e): demand-weighted criticality ---------------------------------
dem = optional_table('21', 'demand_weighted_criticality.csv',
                     '21_demand_weighted_criticality', 'demand_weighted')

if dem is not None and 'demand_weighted_rank' in dem.columns:
    register_lens(
        'demand_weighted', dem, 'stop_id', 'demand_weighted_rank', higher_is_worse=False,
        question='Which stations matter most once the population served is weighted in?',
        source='21_demand_weighted_criticality/tables/demand_weighted_criticality.csv')
else:
    proxy = optional_table('21', 'demand_proxy.csv', '21_demand_weighted_criticality',
                           'demand_weighted (proxy fallback)')
    if proxy is not None:
        print('[fallback] using demand_proxy.demand_weight; this is raw served demand, '
              'not the demand-weighted criticality ranking itself.')
    register_lens(
        'demand_weighted', proxy, 'stop_id', 'demand_weight', higher_is_worse=True,
        question='Which stations serve the most demand (proxy fallback)?',
        source='21_demand_weighted_criticality/tables/demand_proxy.csv')

## 11. עדשה (ו) קריטיוּת בשעת שיא

מחברת 20 מחשבת מחדש מרכזיות בנפרד לכל חלון זמן שנבנה במחברת 19, משום שהרשת ב-08:00 אינה הרשת ב-23:00 - ענפים הפועלים רק בשעת העומס קיימים באחת ולא באחרת. העדשה הזו לוקחת את חלון ה-**peak** בלבד.

בחירת אותו חלון חייבת להיות חסינה לשמות שמחברת 19 בחרה, ולכן הבחירה היא: להעדיף שמות חלון המכילים `peak` אך לא `off`; אם מספר חלונות עומדים בכך, להשתמש ב-`window_summary.csv` של מחברת 19 כדי לקחת את זה עם מירב הנסיעות; ואם אותה טבלה אינה זמינה, לקחת את המועמד הראשון לפי סדר אלפביתי. החלון הנבחר מודפס, ולכן הבחירה לעולם אינה שקטה. בתוך החלון אנו מדרגים לפי `approx_betweenness`, עם נסיגה ל-`weighted_degree` אם עמודת ה-betweenness אינה קיימת.

In [ ]:
# --- Lens (f): peak-hour criticality ---------------------------------------
peak = optional_table('20', 'peak_vs_offpeak_centrality.csv', '20_dynamic_resilience',
                      'peak_hour')
peak_window = None

if peak is not None and 'window' in peak.columns:
    windows = sorted({str(w) for w in peak['window'].dropna()})
    named = [w for w in windows if 'peak' in w.lower() and 'off' not in w.lower()]
    candidates = named or windows
    ws_path = find_artifact('19', 'window_summary.csv')
    if len(candidates) > 1 and ws_path is not None:
        ws = _read(ws_path)
        ws['window'] = ws['window'].astype(str)
        busiest = ws[ws['window'].isin(candidates)]
        if 'trips' in busiest.columns and len(busiest):
            candidates = [busiest.sort_values('trips', ascending=False).iloc[0]['window']]
    peak_window = str(candidates[0])
    print(f'windows available: {windows}')
    print(f'peak window selected: "{peak_window}"')
    peak = peak[peak['window'].astype(str) == peak_window].copy()

peak_metric = 'weighted_degree'
if peak is not None and 'approx_betweenness' in peak.columns:
    peak_metric = 'approx_betweenness'

register_lens(
    'peak_hour', peak, 'stop_id', peak_metric, higher_is_worse=True,
    question='Which stations are critical specifically during the peak window?',
    source='20_dynamic_resilience/tables/peak_vs_offpeak_centrality.csv'
           + (f' [window={peak_window}]' if peak_window else ''))

## 12. עדשה (ז) עלות זמן הנוסעים בסגירת התחנה

מחברת 22 סוגרת תחנה אחת בכל פעם על גרף זמני הנסיעה ומודדת בכמה מתארכת הנסיעה של כל היתר. זהו הדבר הקרוב ביותר בפרויקט למדד השפעה אמיתי, משום שהוא נקוב בשניות של זמן נוסע ולא ביחידות גרף.

יש כאן החלטת מידול אחת שיש להצהיר עליה בגלוי. `mean_detour_seconds` הוא ממוצע על פני הזוגות ש**עדיין ברי-הגעה** לאחר הסגירה. תחנה שהסרתה מנתקת אזור שלם עשויה אפוא לרשום מעקף ממוצע *נמוך*, פשוט משום שהנסיעות שנפגעו הכי קשה נשמטו מהממוצע כליל - המדד מתגמל כישלון קטסטרופלי. כדי למנוע את ההיפוך הזה אנו מחייבים במפורש על חוסר יכולת הגעה:

```
closure_cost_seconds = mean_detour_seconds + UNREACHABLE_PENALTY_SECONDS * (1 - reachable_share)
```

`UNREACHABLE_PENALTY_SECONDS = 3600` אומר "התייחס לנסיעה שהפכה בלתי אפשרית כאילו עלתה שעה". המספר הזה הוא **הנחה, לא מדידה** - העלות האמיתית של נסיעה בלתי אפשרית אינה חסומה. הוא קבוע בעל שם דווקא כדי שניתן יהיה לבחון את השפעתו: הגדילו אותו ותחנות מנתקות ישתלטו על העדשה, קבעו אותו ל-0 ותקבלו את המעקף הממוצע הגולמי (וההפוך). אם מחברת 22 אינה מייצאת `reachable_share`, נעשה שימוש במעקף הממוצע הגולמי וההסתייגות שלעיל נותרת בעינה ללא הקלה.

In [ ]:
# --- Lens (g): passenger-time cost under closure ---------------------------
rr = optional_table('22', 'rerouting_results.csv', '22_rerouting_analysis',
                    'closure_time_cost')

if rr is not None and 'mean_detour_seconds' in rr.columns:
    rr = rr.copy()
    cost = pd.to_numeric(rr['mean_detour_seconds'], errors='coerce').fillna(0.0)
    if 'reachable_share' in rr.columns:
        lost = 1.0 - pd.to_numeric(rr['reachable_share'], errors='coerce').clip(0, 1).fillna(1.0)
        cost = cost + UNREACHABLE_PENALTY_SECONDS * lost
        print(f'unreachable demand charged at {UNREACHABLE_PENALTY_SECONDS:,} s '
              '(an assumption, see the markdown above)')
    else:
        print('[caveat  ] rerouting_results.csv has no reachable_share column, so the '
              'mean detour is used raw - stations that disconnect the network are '
              'under-rated by this lens.')
    rr['closure_cost_seconds'] = cost

register_lens(
    'closure_time_cost', rr, 'removed_stop', 'closure_cost_seconds', higher_is_worse=True,
    question='Which closures cost passengers the most travel time?',
    source='22_rerouting_analysis/tables/rerouting_results.csv')

## 13. עדשה (ח) מעבר רב-אופני

מחברת 17 סופרת כמה אופנים נבדלים (אוטובוס, רכבת, רכבת קלה, טרולייבוס, רכבל, ושירות מבוסס-ביקוש) נפגשים בכל תחנה. תחנה שבה מתחלפים ארבעה אופנים היא קריטית באופן שונה מתחנה בעלת betweenness גבוה: אובדנה אינו רק מאריך מסלולים, אלא שובר את המעבר שמאפשר בכלל נסיעה רב-שלבית, ולרוב אין מקום שני בקרבת מקום שבו אותם שני אופנים נפגשים.

`n_modes` הוא מספר שלם קטן, ולכן לעדשה הזו יש תיקואים כבדים מאוד - רוב התחנות הן 1, כמה מאות הן 2, וקומץ הן 3 ומעלה. תיקואי דירוג מטופלים באמצעות `method='min'`, אך המשמעות המעשית היא שה-"top 50" שלה הוא פרוסה שרירותית מתוך קבוצת תיקו גדולה. יש לקרוא את ספירות החפיפה שלה מתוך מודעות לכך; מקדם המתאם אינפורמטיבי יותר מספירת ה-top-50 עבור עדשה זו.

In [ ]:
# --- Lens (h): multimodal interchange --------------------------------------
hubs = optional_table('17', 'transfer_hubs.csv', '17_multimodal_transfer_hubs',
                      'multimodal')

if hubs is not None and 'n_modes' in hubs.columns:
    counts = pd.to_numeric(hubs['n_modes'], errors='coerce').value_counts().sort_index()
    print('modes served -> number of stops:')
    print(counts.to_string())

register_lens(
    'multimodal', hubs, 'stop_id', 'n_modes', higher_is_worse=True,
    question='Which stations are the interchange point between several modes?',
    source='17_multimodal_transfer_hubs/tables/transfer_hubs.csv')

## 14. מה קיבלנו בסופו של דבר

לפני כל השוואה, אנו רושמים במדויק אילו עדשות קיימות בהרצה זו וכמה מהרשת כל אחת מהן מסוגלת לראות. `lens_coverage.csv` הוא רשומת הכנות של המחברת: כל מתאם וכל דירוג ברשימה המקוצרת שלהלן חייבים להיקרא מולו, משום שעדשה המכסה 3,000 תחנות ועדשה המכסה 30,000 אינן אינפורמטיביות באותה מידה אפילו כשהן מפיקות מספר שנראה זהה.

ההשוואה זקוקה לשתי עדשות לכל הפחות כדי שתהיה לה משמעות, ולכן התא נעצר עם הודעה ברורה אם שרדו פחות משתיים.

In [ ]:
# --- Lens inventory --------------------------------------------------------
if len(LENSES) < 2:
    raise RuntimeError(
        f'only {len(LENSES)} lens/lenses could be built, so there is nothing to compare. '
        'Run notebooks 03, 04 and 05 first (04 is mandatory), then re-run this notebook.')

lens_names = list(LENSES)
coverage = (pd.DataFrame([LENS_META[n] for n in lens_names])
            .sort_values('stations_evaluated', ascending=False)
            .reset_index(drop=True))
coverage.to_csv(TABLES / 'lens_coverage.csv', index=False, encoding='utf-8-sig')

print(f'{len(lens_names)} lenses available: {lens_names}')
if MISSING_LENSES:
    print('\nlenses NOT available in this run:')
    for label, notebook, what in MISSING_LENSES:
        print(f'   {label:<32} needs {notebook}  (missing: {what})')
else:
    print('all eight lenses were available.')

coverage[['lens', 'metric', 'stations_evaluated', 'coverage_share_of_universe']]

## 15. הטבלה הארוכה: כל תחנה תחת כל עדשה

`critical_station_lenses.csv` הוא פלט החוזה של שלב זה וחומר הגלם לכל מה שבא אחריו: שורה אחת לכל זוג (תחנה, עדשה) עם הדירוג וציון הקריטיוּת, בפורמט ארוך, כך שהוספת עדשה תשיעית בהמשך משנה את מספר השורות ולא את הסכמה. יש לזכור ש-`score` נשמע למוסכמה הכלל-מחברתית - גבוה יותר הוא תמיד קריטי יותר - ולכן עבור עדשה שהמדד המקורי שלה היה מספר דירוג, הציון הנשמר הוא ההיפוך שלו. עמודת `rank` היא זו הקריאה לאדם: 1 היא התחנה הקריטית ביותר תחת אותה עדשה.

In [ ]:
# --- Long table: one row per (station, lens) -------------------------------
long_parts = []
for name in lens_names:
    part = LENSES[name][['stop_id', 'rank', 'score']].copy()
    part.insert(1, 'lens', name)
    long_parts.append(part)

lenses_long = pd.concat(long_parts, ignore_index=True)
lenses_long['stop_name'] = lenses_long['stop_id'].map(NAME_BY_ID)
lenses_long = (lenses_long[['stop_id', 'stop_name', 'lens', 'rank', 'score']]
               .sort_values(['lens', 'rank'])
               .reset_index(drop=True))
lenses_long.to_csv(TABLES / 'critical_station_lenses.csv', index=False, encoding='utf-8-sig')

print(f'critical_station_lenses.csv: {len(lenses_long):,} rows '
      f'({lenses_long["stop_id"].nunique():,} distinct stations x {len(lens_names)} lenses)')
lenses_long.groupby('lens').head(3).head(24)

## 16. האם העדשות מסכימות? מתאם Spearman וחפיפת top-K

עבור כל זוג עדשות אנו מחשבים שני דברים שונים מאוד, וזאת במתכוון:

* **מתאם דירוגים Spearman** על פני התחנות ש*שתי* העדשות העריכו. הוא עונה על "האם שני הסדרים האלה מסכימים על פני הרשת כולה?". הוא מחושב pairwise-complete, ולכן זוג המערב עדשה צרה מחושב על אותה חפיפה צרה בלבד - וזו הסיבה ש-`n_common_stations` מיוצא לצידו ב-`lens_agreement_detail.csv`, וזו הסיבה שזוגות עם פחות מ-`MIN_COMMON_FOR_RHO` תחנות משותפות נותרים ריקים במקום להיות מדווחים כמקריות.
* **חפיפת top-K**: כמה מאותן תחנות מופיעות ב-top 50 של שתי העדשות. זו התשובה לשאלה שמתכנן באמת שואל, שאינה "האם הסדרים דומים באופן כללי" אלא "האם שתי ההגדרות מפנות אותי לאותו קומץ תחנות". השתיים יכולות להתפצל בחדות: שתי עדשות יכולות להיות במתאם של rho = 0.8 על פני 30,000 תחנות ועדיין לחלוק רק קומץ שמות ממש בפסגה, משום שמתאם דירוגים נשלט על ידי האמצע העצום והבלתי מעניין של ההתפלגות.

נעשה שימוש ב-Spearman ולא ב-Pearson משום שציוני העדשות נתונים ביחידות בלתי תואמות (שניות, מטרים, נסיעות, ספירות אופנים) וכמה מהם מוטים מאוד; רק הסדר הוא בר-השוואה.

In [ ]:
# --- Pairwise agreement between lenses -------------------------------------
topk_sets = {n: set(LENSES[n].sort_values('rank')['stop_id'].head(TOP_K))
             for n in lens_names}

pair_rows = []
for i, a in enumerate(lens_names):
    for b in lens_names[i + 1:]:
        da = LENSES[a][['stop_id', 'score']].rename(columns={'score': 'score_a'})
        db = LENSES[b][['stop_id', 'score']].rename(columns={'score': 'score_b'})
        both = da.merge(db, on='stop_id', how='inner')
        if len(both) >= MIN_COMMON_FOR_RHO:
            rho = float(spearmanr(both['score_a'], both['score_b'])[0])
        else:
            rho = float('nan')
        pair_rows.append({
            'lens_a': a,
            'lens_b': b,
            'spearman_rho': round(rho, 4),
            'top50_overlap': int(len(topk_sets[a] & topk_sets[b])),
            'n_common_stations': int(len(both)),
        })

agreement = pd.DataFrame(pair_rows)
# contracted file: exactly the four agreed columns
agreement[['lens_a', 'lens_b', 'spearman_rho', 'top50_overlap']].to_csv(
    TABLES / 'lens_agreement.csv', index=False, encoding='utf-8-sig')
# diagnostic file: the same pairs plus the overlap size the rho was computed on
agreement.to_csv(TABLES / 'lens_agreement_detail.csv', index=False, encoding='utf-8-sig')

thin = agreement[agreement['spearman_rho'].isna()]
if len(thin):
    print(f'{len(thin)} lens pair(s) share fewer than {MIN_COMMON_FOR_RHO} stations; '
          'their correlation is left blank rather than reported.')
agreement.sort_values('top50_overlap', ascending=False)

## 17. מפות חום של ההסכמה

שתי מטריצות ריבועיות שנבנו מטבלת הזוגות שלעיל. השמאלית היא מטריצת Spearman על סקאלה מסתעפת אדום-כחול המקובעת ל-[-1, 1], כך שהצבע בר-השוואה בין הרצות ותא ריק פירושו "חפיפה מועטה מכדי לקבוע" ולא "מתאם אפס". הימנית סופרת חברים משותפים ב-top 50; האלכסון שלה הוא גודל קבוצת ה-top-K של כל עדשה, שהוא בדרך כלל 50 אך יכול להיות קטן יותר עבור עדשה המכסה פחות מ-50 תחנות.

קריאתן יחד היא כל עניינו של הסעיף. מתאם גבוה לצד חפיפת top-50 נמוכה משמעו שהעדשות מסכימות באופן כללי לגבי המדינה אך חלוקות לגבי הקצוות - והקצוות הם אלה שמקבלים תקציבי תחזוקה.

In [ ]:
# --- Agreement heatmaps ----------------------------------------------------
rho_m = pd.DataFrame(np.nan, index=lens_names, columns=lens_names, dtype=float)
ovl_m = pd.DataFrame(0, index=lens_names, columns=lens_names, dtype=int)
for _, r in agreement.iterrows():
    a, b = r['lens_a'], r['lens_b']
    rho_m.loc[a, b] = rho_m.loc[b, a] = r['spearman_rho']
    ovl_m.loc[a, b] = ovl_m.loc[b, a] = int(r['top50_overlap'])
for n in lens_names:
    rho_m.loc[n, n] = 1.0
    ovl_m.loc[n, n] = len(topk_sets[n])

fig, ax = plt.subplots(figsize=(1.6 * len(lens_names) + 3, 1.3 * len(lens_names) + 2))
sns.heatmap(rho_m, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, center=0,
            linewidths=0.5, cbar_kws={'label': 'Spearman rho'}, ax=ax)
ax.set_title('Agreement between criticality lenses\n'
             '(rank correlation on the stations both lenses evaluated)')
plt.tight_layout()
plt.savefig(FIGURES / 'lens_spearman_heatmap.png', dpi=FIG_DPI)
plt.show()

fig, ax = plt.subplots(figsize=(1.6 * len(lens_names) + 3, 1.3 * len(lens_names) + 2))
sns.heatmap(ovl_m, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': f'stations shared in the top {TOP_K}'}, ax=ax)
ax.set_title(f'Top-{TOP_K} overlap between criticality lenses\n'
             '(diagonal = size of each lens own top set)')
plt.tight_layout()
plt.savefig(FIGURES / 'lens_top_overlap_heatmap.png', dpi=FIG_DPI)
plt.show()

## 18. פרופיל לכל תחנה: קריטית באופן חסין מול תלוית-עדשה

כעת אנו מבצעים pivot לטבלה הארוכה לכדי שורה אחת לתחנה ומצרפים שלושה מספרי סיכום:

* **`n_lenses_evaluated`** - כמה עדשות הצליחו בכלל לנקד את התחנה הזו. זהו מכנה הכיסוי, ובלעדיו שני המספרים הבאים בלתי קריאים.
* **`n_lenses_topk`** - בכמה מה-top 50 של העדשות התחנה מופיעה. זהו *מונה הראיות* ומפתח המיון הראשי. כאן, תחנה שעדשה מעולם לא העריכה נספרת כראוי כ"לא סומנה על ידי אותה עדשה", משום שהטענה המופנית למתכנן היא "כמה הגדרות בלתי תלויות של קריטיוּת בוחרות את התחנה הזו".
* **`consensus_mean_pct`** - האחוזון הממוצע על פני העדשות שכן העריכו אותה, המשמש רק כשובר תיקו. מספר זה הוא *מותנה-כיסוי* ואסור לקרוא אותו כציון כולל: תחנה שנצפתה על ידי עדשה צרה יחידה יכולה לקבל כאן 0.99 מכוחה של דעה אחת, וזו בדיוק הסיבה שהרשימה המקוצרת שלהלן דורשת גם `n_lenses_evaluated >= MIN_LENSES_FOR_SHORTLIST`.

שני מצבי הכשל שאנו מפרידים ביניהם הם: **קריטיוּת חסינה** (כמה הגדרות בלתי קשורות מצביעות באופן בלתי תלוי על אותה תחנה) ו**תלות-עדשה** (בדיוק עדשה אחת מסמנת אותה בעוד אחרות שכן בחנו אותה לא סימנו - לרוב תוצר לוואי של הגדרת אותה עדשה, ולעיתים נקודה עיוורת אמיתית בכל האחרות).

In [ ]:
# --- Wide per-station profile ----------------------------------------------
rank_wide = pd.concat([LENSES[n].set_index('stop_id')['rank'].rename(n)
                       for n in lens_names], axis=1)
pct_wide = pd.concat([LENSES[n].set_index('stop_id')['pct'].rename(n)
                      for n in lens_names], axis=1)
rank_wide.index.name = 'stop_id'
pct_wide.index.name = 'stop_id'
flags = rank_wide.le(TOP_K)          # NaN ranks compare False, which is what we want

prof = pd.DataFrame(index=pct_wide.index)
prof['n_lenses_evaluated'] = pct_wide.notna().sum(axis=1).astype(int)
prof['n_lenses_topk'] = flags.sum(axis=1).astype(int)
prof['consensus_mean_pct'] = pct_wide.mean(axis=1).round(4)
prof['lenses_topk'] = flags.apply(
    lambda row: ';'.join([c for c in flags.columns if bool(row[c])]), axis=1)
prof = prof.reset_index()

prof = prof.merge(rank_wide.add_prefix('rank_').reset_index(), on='stop_id', how='left')
prof = prof.merge(pct_wide.add_prefix('pct_').reset_index(), on='stop_id', how='left')
prof = prof.merge(spine, on='stop_id', how='left')


def reason_text(row):
    """Human-readable justification: which lenses flagged this station, and at what rank."""
    bits = [f'{n} (#{int(row["rank_" + n])})'
            for n in lens_names
            if pd.notna(row['rank_' + n]) and row['rank_' + n] <= TOP_K]
    return '; '.join(bits) if bits else 'in no lens top-' + str(TOP_K)


prof['reasons'] = prof.apply(reason_text, axis=1)
prof = prof.sort_values(['n_lenses_topk', 'consensus_mean_pct'],
                        ascending=[False, False]).reset_index(drop=True)
prof.to_csv(TABLES / 'station_lens_profile.csv', index=False, encoding='utf-8-sig')

dist = prof['n_lenses_topk'].value_counts().sort_index()
print('stations flagged by exactly k lenses:')
for k, v in dist.items():
    print(f'   k = {k}: {v:,} stations')
print(f'\nflagged by at least one lens: {int((prof["n_lenses_topk"] > 0).sum()):,}')
print(f'flagged by two or more lenses: {int((prof["n_lenses_topk"] >= 2).sum()):,}')

## 19. עד כמה הראיות מרוכזות?

תרשים עמודות של מספר התחנות המסומנות על ידי עדשה אחת בדיוק, שתיים בדיוק, וכן הלאה. צורת התרשים הזה היא תוצאת הכותרת של המחברת. אם כמעט כל תחנה מסומנת מסומנת על ידי עדשה אחת בדיוק, אזי "קריטית" היא בעיקרה תכונה של המדידה ולא של התחנה, ואין להציג שום דירוג יחיד בפרויקט הזה כ*התשובה*. אם קבוצה נראית לעין מסומנת על ידי שלוש עדשות ומעלה, אותן תחנות הן הגרעין החסין - והערך המרבי האפשרי הוא מספר העדשות הזמינות בהרצה זו, אותו התרשים מציין בכותרתו.

In [ ]:
# --- Distribution of evidence across lenses --------------------------------
flagged = prof[prof['n_lenses_topk'] > 0]
counts = flagged['n_lenses_topk'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([str(k) for k in counts.index], counts.to_numpy(), color='#2563eb')
for bar, val in zip(bars, counts.to_numpy()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:,}',
            ha='center', va='bottom', fontsize=10)
ax.set_xlabel(f'Number of lenses placing the station in their top {TOP_K}')
ax.set_ylabel('Number of stations')
ax.set_title(f'How many stations are critical under how many lenses\n'
             f'({len(lens_names)} lenses available in this run)')
ax.margins(y=0.14)
plt.tight_layout()
plt.savefig(FIGURES / 'stations_by_lens_count.png', dpi=FIG_DPI)
plt.show()

## 20. הרשימה המקוצרת

התוצר המעשי. תחנה מגיעה לרשימה המקוצרת רק אם הוערכה על ידי `MIN_LENSES_FOR_SHORTLIST` עדשות לפחות (כך ששום דבר אינו נכנס מכוח דעה צרה יחידה) ומופיעה ב-top 50 של עדשה אחת לפחות. סדר המיון הוא: מספר העדשות המסמנות אותה, ולאחר מכן האחוזון הממוצע כשובר תיקו.

כל שורה נושאת מחרוזת `reasons` המפרטת בדיוק אילו עדשות סימנו אותה ובאיזה דירוג, כך שהרשימה ניתנת לביקורת - מתכנן יכול לראות שתחנה מופיעה כאן משום שהיא cut vertex המבודד 400 תחנות *וגם* צומת המעבר העמוס ביותר במחוזה, לעומת תחנה שנמצאת כאן על סמך betweenness בלבד. עמודות `rank_*` לכל עדשה נשמרות בקובץ המיוצא מאותו טעם עצמו.

זו רשימה מקוצרת, לא תקציב. היא מדרגת תחנות לפי כמות הראיות הבלתי תלויות לכך שהן חשובות - לא לפי עלות תיקון, לא לפי הסתברות כשל, ולא לפי שום דבר שקובץ ה-GTFS אינו יכול לראות (תצורת התחנה, איוש, ספירות נוסעים בפועל).

In [ ]:
# --- Final ranked shortlist ------------------------------------------------
eligible = prof[(prof['n_lenses_evaluated'] >= MIN_LENSES_FOR_SHORTLIST)
                & (prof['n_lenses_topk'] >= 1)].copy()
shortlist = (eligible.sort_values(['n_lenses_topk', 'consensus_mean_pct'],
                                  ascending=[False, False])
             .head(SHORTLIST_SIZE)
             .reset_index(drop=True))
shortlist.insert(0, 'shortlist_rank', np.arange(1, len(shortlist) + 1))

front = ['shortlist_rank', 'stop_id', 'stop_name', 'region', 'lat', 'lon',
         'n_lenses_topk', 'n_lenses_evaluated', 'consensus_mean_pct', 'reasons']
front = [c for c in front if c in shortlist.columns]
rest = [c for c in shortlist.columns if c not in front]
shortlist = shortlist[front + rest]
shortlist.to_csv(TABLES / 'critical_station_shortlist.csv', index=False, encoding='utf-8-sig')

print(f'{len(eligible):,} stations were eligible; the top {len(shortlist)} are exported.')
shortlist[['shortlist_rank', 'stop_name', 'region', 'n_lenses_topk',
           'consensus_mean_pct', 'reasons']].head(TOP_N_FIG)

## 21. מפת חום של פרופיל הרשימה המקוצרת

שורה אחת לכל תחנה ברשימה המקוצרת, עמודה אחת לכל עדשה, צבועה לפי האחוזון של התחנה בתוך אותה עדשה (1.0 = התחנה הקריטית ביותר שאותה עדשה ראתה אי פעם, 0 = הפחותה ביותר). פערים אפורים הם *לא הוערכה* - התחנה נמצאת מחוץ לכיסוי אותה עדשה - והם מצוירים באופן נבדל מציון נמוך אמיתי, משום ששני אלה מציינים דברים הפוכים.

שורה כהה לכל אורכה היא תחנה שכל ההגדרות מסכימות לגביה. שורה עם תא כהה אחד ושאר התאים חיוורים היא תחנה הרוכבת על עדשה יחידה, והיא המקבילה החזותית של טבלת תלויות-העדשה בסעיף 23. שמות התחנות הם בעברית ועוברים דרך patch ה-bidi שהותקן בסעיף 3; מזהה התחנה המספרי מצורף בסופם משום שכמה תחנות חולקות שם.

In [ ]:
# --- Shortlist profile heatmap ---------------------------------------------
top_rows = shortlist.head(TOP_N_FIG).copy()
pct_cols = ['pct_' + n for n in lens_names]
mat = top_rows[pct_cols].astype(float)
mat.columns = lens_names
mat.index = [f'{name} ({sid})' for name, sid
             in zip(top_rows['stop_name'].fillna('?'), top_rows['stop_id'])]

fig, ax = plt.subplots(figsize=(1.5 * len(lens_names) + 6, 0.42 * len(mat) + 3))
sns.heatmap(mat, annot=True, fmt='.2f', cmap='magma_r', vmin=0, vmax=1,
            linewidths=0.4, linecolor='white',
            cbar_kws={'label': 'percentile within the lens (1.0 = most critical)'},
            mask=mat.isna(), ax=ax)
ax.set_facecolor('#d9d9d9')          # grey = the lens never evaluated this station
ax.set_title(f'Top {len(mat)} shortlisted stations under every available lens\n'
             'grey = not evaluated by that lens (not "low score")')
ax.set_xlabel('lens')
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES / 'shortlist_lens_profile.png', dpi=FIG_DPI)
plt.show()

## 22. היכן נמצאת הרשימה המקוצרת

הרשימה המקוצרת משורטטת מעל ענן התחנות המלא, כאשר גודל התחנה וצבעה נקבעים לפי מספר העדשות שסימנו אותה. זהו scatter פשוט של קו רוחב/אורך ולא מפה מוטלת, ולכן יחס הממדים נקבע ל-`1 / cos(mean latitude)` כדי לשמור על צורת המדינה קרובה לנכון במקום מתוחה לרוחב.

לגאוגרפיה יש חשיבות לפרשנות: אם הרשימה המקוצרת מתקבצת כולה במסדרון תל אביב - חיפה, אזי הסינתזה בעיקר מגלה מחדש היכן הרשת צפופה ביותר, והפריפריה - שבה לכשל בודד יש הרבה פחות חלופות - זוכה לתת-ייצוג על ידי כל העדשות בבת אחת.

In [ ]:
# --- Map of the shortlist --------------------------------------------------
base = spine.dropna(subset=['lat', 'lon'])
pts = shortlist.dropna(subset=['lat', 'lon'])

fig, ax = plt.subplots(figsize=(8, 11))
ax.scatter(base['lon'], base['lat'], s=1.5, color='#cbd5e1', alpha=0.55, linewidths=0)
if len(pts):
    sc = ax.scatter(pts['lon'], pts['lat'], c=pts['n_lenses_topk'],
                    s=40 + 55 * pts['n_lenses_topk'], cmap='plasma_r',
                    edgecolor='black', linewidths=0.5, zorder=3)
    cbar = plt.colorbar(sc, ax=ax, fraction=0.035)
    cbar.set_label(f'lenses placing the station in their top {TOP_K}')
    ax.set_aspect(1 / np.cos(np.deg2rad(float(base['lat'].mean()))))
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title(f'Nationally critical stations shortlist (n = {len(pts)})\n'
             'grey = all other active stops')
plt.tight_layout()
plt.savefig(FIGURES / 'shortlist_map.png', dpi=FIG_DPI)
plt.show()

if 'region' in shortlist.columns:
    print('shortlist by region:')
    print(shortlist['region'].fillna('Unknown').value_counts().to_string())

## 23. תחנות תלויות-עדשה: תוצרי הלוואי

תמונת הראי של הרשימה המקוצרת. אלו תחנות שעדשה אחת בדיוק מציבה ב-top 50 שלה **בעוד שלפחות עדשה אחת אחרת בחנה אותן ולא עשתה כן**. התנאי השני הוא שהופך את הממצא למשמעותי: תחנה שסומנה על ידי עדשה אחת ופשוט בלתי נראית לכל האחרות היא פער כיסוי, ולא אי-הסכמה.

רובן הן תוצרי לוואי של הגדרות, וכדאי לנקוב בשמן משום שכל אחת מהן היא אזהרה לגבי העדשה שהפיקה אותה - cut vertex על ענף מבוי סתום ששום נפח נוסעים אינו מצדיק, תחנה עם `nearest_alt_m` עצום משום שהיא ניצבת לבדה במדבר, תחנה רב-אופנית שהיא רב-אופנית רק משום ששני סוגי מסלול במקרה חולקים אותה מדרכה. הספירות לכל עדשה מלמדות אילו עדשה היא האידיוסינקרטית ביותר: העדשה התורמת את מירב הסימונים חד-עדשתיים היא זו שדירוגה ראוי פחות מכול לשמש בפני עצמו.

In [ ]:
# --- Stations flagged by exactly one lens ----------------------------------
single = prof[(prof['n_lenses_topk'] == 1)
              & (prof['n_lenses_evaluated'] >= 2)].copy()
single['only_lens'] = single['lenses_topk']
single_out = single[['stop_id', 'stop_name', 'region', 'lat', 'lon', 'only_lens',
                     'n_lenses_evaluated', 'consensus_mean_pct', 'reasons']]
single_out = single_out.sort_values(['only_lens', 'consensus_mean_pct'],
                                    ascending=[True, False])
single_out.to_csv(TABLES / 'lens_specific_stations.csv', index=False, encoding='utf-8-sig')

per_lens = single['only_lens'].value_counts().reindex(lens_names).fillna(0).astype(int)
print(f'{len(single):,} stations are flagged by exactly one lens while at least one '
      'other lens evaluated them and did not flag them.')

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(per_lens.index.astype(str), per_lens.to_numpy(), color='#dc2626')
for bar, val in zip(bars, per_lens.to_numpy()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:,}',
            ha='center', va='bottom', fontsize=10)
ax.set_xlabel('lens')
ax.set_ylabel('stations flagged by this lens alone')
ax.set_title('Lens-specific flags: how idiosyncratic is each definition of critical?')
plt.setp(ax.get_xticklabels(), rotation=25, ha='right')
ax.margins(y=0.14)
plt.tight_layout()
plt.savefig(FIGURES / 'lens_specific_counts.png', dpi=FIG_DPI)
plt.show()

single_out.head(15)

## 24. רשומת הסיכום

קובץ JSON יחיד ובו מספרי הכותרת, העדשות שהיו זמינות והעדשות שלא, הממוצע והטווח של ההסכמה הזוגית, וראש הרשימה המקוצרת. נכתב אחרון כדי שיתאר תמיד את ההרצה שזה עתה התרחשה - ובכלל זה, וזה חשוב, אילו עדשות היו חסרות, כך שקורא ה-JSON לבדו לא יוכל לטעות ולחשוב שהרצה של ארבע עדשות היא הרצה של שמונה.

In [ ]:
# --- Summary JSON ----------------------------------------------------------
rho_vals = agreement['spearman_rho'].dropna()
top_rows_json = shortlist.head(10)[['shortlist_rank', 'stop_id', 'stop_name',
                                    'n_lenses_topk', 'consensus_mean_pct', 'reasons']]

summary = {
    'stations_in_universe': int(len(spine)),
    'top_k_definition': int(TOP_K),
    'lenses_available': lens_names,
    'lenses_missing': [{'lens': a, 'needs': b, 'missing': c} for a, b, c in MISSING_LENSES],
    'lens_coverage': {n: LENS_META[n]['stations_evaluated'] for n in lens_names},
    'lens_pairs_compared': int(len(agreement)),
    'spearman_rho_mean': round(float(rho_vals.mean()), 4) if len(rho_vals) else None,
    'spearman_rho_min': round(float(rho_vals.min()), 4) if len(rho_vals) else None,
    'spearman_rho_max': round(float(rho_vals.max()), 4) if len(rho_vals) else None,
    'top_k_overlap_mean': round(float(agreement['top50_overlap'].mean()), 2),
    'stations_flagged_by_any_lens': int((prof['n_lenses_topk'] > 0).sum()),
    'stations_flagged_by_two_or_more': int((prof['n_lenses_topk'] >= 2).sum()),
    'stations_flagged_by_three_or_more': int((prof['n_lenses_topk'] >= 3).sum()),
    'stations_flagged_by_exactly_one': int(len(single)),
    'max_lenses_agreeing_on_one_station': int(prof['n_lenses_topk'].max()),
    'shortlist_size': int(len(shortlist)),
    'unreachable_penalty_seconds': int(UNREACHABLE_PENALTY_SECONDS),
    'articulation_severity_exact': bool(AP_SEVERITY_EXACT),
    'shortlist_top10': top_rows_json.to_dict(orient='records'),
}

with open(STAGE / 'lens_synthesis_summary.json', 'w', encoding='utf-8') as fh:
    json.dump(summary, fh, ensure_ascii=False, indent=2)

print(json.dumps({k: v for k, v in summary.items() if k != 'shortlist_top10'},
                 ensure_ascii=False, indent=2))
print('\nfiles written under', STAGE)
for p in sorted(STAGE.rglob('*')):
    if p.is_file():
        print('   ', p.relative_to(STAGE))

## מסקנות

* **"קריטית" אינה תכונה אחת, והמחברת הזו היא ההוכחה.** שמונה הגדרות היו זמינות עקרונית; כל אחת מהן מפיקה top 50 שונה, ומטריצת Spearman בסעיף 17 מראה כמה רחוקות הן זו מזו. שתי העדשות המגיעות מאותה טבלה (betweenness ונפח שירות) מסכימות ביותר, וזו בדיקת שפיות ולא ממצא; המספרים המעניינים הם התאים שמחוץ לאלכסון, המחברים עדשות שנבנו מנתונים שונים - מבנה, מרחק הליכה, שניות נוסע, ספירות אופנים.
* **מתאם דירוגים וחפיפת top-50 מספרים סיפורים שונים, וחפיפת ה-top-50 היא זו שחשובה.** שתי עדשות יכולות להיות במתאם חזק על פני 30,000 תחנות - משום שהן מסכימות שהרוב המכריע של התחנות אינו יוצא דופן - ועדיין לחלוק מעט מאוד שמות ממש בפסגה. כל טענה בפרויקט הזה מהצורה "תחנה X היא הקריטית ביותר במדינה" היא טענה על עדשה אחת, וסעיף 16 מכמת כמה מעט ממנה שורד שינוי הגדרה.
* **גרעין חסין קטן אכן קיים.** התחנות שכמה עדשות בלתי קשורות מציבות באופן בלתי תלוי ב-top 50 שלהן הן התשובה הניתנת להגנה לשאלת המחקר של הפרויקט. הן מיוצאות ב-`critical_station_shortlist.csv` יחד עם הנימוק לכל אחת, והן התחנות היחידות בפרויקט הזה הנתמכות ביותר מסוג ראיה אחד.
* **הסימונים חד-העדשתיים הם ברובם תוצרי לוואי, ונקיבת שמם היא חלק מהתוצאה.** cut vertex על ענף מבוי סתום, תחנה ללא חלופה בהליכה משום ששום דבר אחר אינו בטווח קילומטר ממנה, "צומת רב-אופני" שהוא שני סוגי מסלול החולקים מדרכה - כל אחד מאלה הוא תוצאה אמיתית של הגדרת העדשה שלו ולא פגיעוּת לאומית. `lens_specific_stations.csv` מפרט אותם, ותרשים העמודות בסעיף 23 מראה איזו עדשה היא האידיוסינקרטית ביותר.
* **מגבלות כנות.** (1) הכיסוי אינו שוויוני: עדשת החלופה בהליכה ראתה אי פעם רק כ-10% מהרשת שסוננו מראש, ולכן מתאמיה מחושבים על תת-הקבוצה הזו והיעדרותה מתחנה משמעה "לא נמדד", לא "תקין". (2) `consensus_mean_pct` הוא מותנה-כיסוי ומשמש רק כשובר תיקו, לעולם לא כציון כותרת. (3) עדשת ה-betweenness היא קירוב מבוסס דגימה שזנבו אינו יציב, ולכן הפרשי דירוג קטנים הם רעש. (4) עדשת עלות הסגירה תלויה ב-`UNREACHABLE_PENALTY_SECONDS`, שהיא הנחה לגבי מה עולה נסיעה בלתי אפשרית, ולא מדידה. (5) הכול כאן נגזר מלוח הזמנים: קובץ ה-GTFS אינו מכיל ספירות נוסעים, ולכן "ביקוש" הוא תמיד פרוקסי אוכלוסייה וכל אמירה על נוסעים יורשת זאת.
* **מה מתכנן צריך לקחת מכאן.** אין לפעול לפי שום דירוג יחיד בפרויקט הזה. יש לפעול לפי החיתוך - וכאשר תחנה מסומנת על ידי עדשה אחת בלבד, יש להתייחס לכך כאל שאלה לבירור ולא כאל תשובה, משום שהעדשה שסימנה אותה מספרת לך משהו ספציפי על *מדוע* היא עשויה להיות חשובה.